# L17: 对话 Agent 实现

> 构建一个流畅的对话式 AI 助手

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional
from datetime import datetime

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.llm import LLMFactory, Message
from backend.memory.short_term import ShortTermMemory
from backend.memory.long_term import LongTermMemory
print("✓ 模块导入成功")

## 17.1 对话 Agent 的核心要素

### 什么是对话 Agent？

**对话 Agent** = 能够进行多轮自然对话的 AI 助手

```
用户: 你好
Agent: 你好！有什么可以帮助你的吗？
用户: 我叫小明
Agent: 你好小明！很高兴认识你。
用户: 我刚才说我叫什么？
Agent: 你说你叫小明。  ← 记住了之前的信息
```

### 核心能力

| 能力 | 说明 |
|------|------|
| **上下文管理** | 维护对话历史 |
| **意图识别** | 理解用户想要什么 |
| **实体提取** | 识别关键信息（人名、地点等）|
| **状态跟踪** | 跟踪对话进程 |
| **自然回复** | 生成流畅的回复 |
| **记忆管理** | 记住重要信息 |

## 17.2 简单对话 Agent 实现

In [ ]:
class ConversationalAgent:
    """
    基础对话 Agent
    """
    
    def __init__(
        self,
        llm,
        system_prompt: str = "你是一个友好的 AI 助手。",
        max_history: int = 20
    ):
        self.llm = llm
        self.system_prompt = system_prompt
        self.memory = ShortTermMemory(max_messages=max_history)
    
    async def chat(self, user_message: str) -> str:
        """
        与用户对话
        """
        # 添加用户消息到历史
        self.memory.add_message(Message(role="user", content=user_message))
        
        # 构建消息列表
        messages = [Message(role="system", content=self.system_prompt)]
        messages.extend(self.memory.get_messages())
        
        # 调用 LLM
        response = await self.llm.chat(messages=messages)
        
        # 添加助手回复到历史
        self.memory.add_message(Message(role="assistant", content=response.content))
        
        return response.content
    
    def get_history(self) -> List[Message]:
        """获取对话历史"""
        return self.memory.get_messages()
    
    def clear_history(self):
        """清空对话历史"""
        self.memory.clear()

print("=== ConversationalAgent 类已定义 ===\n")
print("方法:")
print("  - chat(user_message): 与用户对话")
print("  - get_history(): 获取对话历史")
print("  - clear_history(): 清空对话历史")

## 17.3 增强版：带记忆的对话 Agent

In [ ]:
class EnhancedConversationalAgent:
    """
    增强版对话 Agent - 支持长期记忆
    """
    
    def __init__(
        self,
        llm,
        system_prompt: str = "你是一个友好的 AI 助手。",
        long_term_memory: LongTermMemory = None,
        user_id: str = None
    ):
        self.llm = llm
        self.system_prompt = system_prompt
        self.short_term = ShortTermMemory()
        self.long_term = long_term_memory
        self.user_id = user_id
    
    async def _retrieve_memories(self, query: str) -> List[str]:
        """
        从长期记忆中检索相关信息
        """
        if not self.long_term:
            return []
        
        try:
            results = await self.long_term.search(
                query=query,
                top_k=3,
                filter={"user_id": self.user_id} if self.user_id else None
            )
            return [r["content"] for r in results]
        except:
            return []
    
    async def _store_important_info(self, content: str):
        """
        将重要信息存入长期记忆
        """
        if not self.long_term:
            return
        
        # 简单判断：包含特定关键词
        important_keywords = ["记住", "名字", "叫", "喜欢", "重要"]
        if any(kw in content for kw in important_keywords):
            await self.long_term.add(
                content=content,
                metadata={
                    "user_id": self.user_id,
                    "timestamp": datetime.now().isoformat()
                }
            )
    
    async def chat(self, user_message: str) -> str:
        """
        与用户对话（增强版）
        """
        # 检索相关记忆
        relevant_memories = await self._retrieve_memories(user_message)
        
        # 构建增强的 System Prompt
        enhanced_prompt = self.system_prompt
        if relevant_memories:
            memory_context = "\n\n相关记忆:\n" + "\n".join(relevant_memories)
            enhanced_prompt += memory_context
        
        # 添加用户消息
        self.short_term.add_message(Message(role="user", content=user_message))
        
        # 构建消息列表
        messages = [Message(role="system", content=enhanced_prompt)]
        messages.extend(self.short_term.get_messages())
        
        # 调用 LLM
        response = await self.llm.chat(messages=messages)
        
        # 存储重要信息
        await self._store_important_info(user_message)
        
        # 添加回复到历史
        self.short_term.add_message(Message(role="assistant", content=response.content))
        
        return response.content

print("=== EnhancedConversationalAgent 类已定义 ===\n")
print("新增方法:")
print("  - _retrieve_memories(): 从长期记忆检索")
print("  - _store_important_info(): 存储重要信息")

## 17.4 对话状态管理

### 状态机设计

In [ ]:
from enum import Enum

class DialogState(Enum):
    """对话状态"""
    GREETING = "greeting"           # 问候
    QUESTION = "question"           # 提问
    TASK_EXECUTION = "task"        # 任务执行
    CLOSING = "closing"             # 结束

class DialogStateManager:
    """
    对话状态管理器
    """
    
    def __init__(self):
        self.current_state = DialogState.GREETING
        self.state_data: Dict[str, Any] = {}
        self.state_transitions = {
            DialogState.GREETING: [DialogState.QUESTION, DialogState.TASK_EXECUTION],
            DialogState.QUESTION: [DialogState.QUESTION, DialogState.CLOSING],
            DialogState.TASK_EXECUTION: [DialogState.QUESTION, DialogState.CLOSING],
            DialogState.CLOSING: [DialogState.GREETING],  # 可以重新开始
        }
    
    def transition_to(self, new_state: DialogState) -> bool:
        """
        转移到新状态
        """
        if new_state in self.state_transitions[self.current_state]:
            self.current_state = new_state
            return True
        return False
    
    def set_data(self, key: str, value: Any):
        """设置状态数据"""
        self.state_data[key] = value
    
    def get_data(self, key: str, default=None):
        """获取状态数据"""
        return self.state_data.get(key, default)

# 演示状态管理
def demo_state_manager():
    manager = DialogStateManager()
    
    print("=== 对话状态管理演示 ===\n")
    print(f"初始状态: {manager.current_state.value}")
    
    # 状态转移
    print(f"转移到 QUESTION: {manager.transition_to(DialogState.QUESTION)}")
    print(f"当前状态: {manager.current_state.value}")
    
    # 保存数据
    manager.set_data("topic", "Python 编程")
    print(f"保存主题: {manager.get_data('topic')}")
    
    print(f"转移到 CLOSING: {manager.transition_to(DialogState.CLOSING)}")
    print(f"当前状态: {manager.current_state.value}")

demo_state_manager()

## 17.5 意图识别

### 简单意图分类

In [ ]:
class IntentClassifier:
    """
    简单的意图分类器
    """
    
    INTENTS = {
        "greeting": ["你好", "嗨", "hello", "hi", "早上好", "晚上好"],
        "goodbye": ["再见", "拜拜", "bye", "晚安", "回见"],
        "question": ["什么", "如何", "怎么", "为什么", "哪", "谁", "?"] ,
        "request": ["请", "帮我", "能否", "可以", "能否"],
        "compliment": ["谢谢", "感谢", "不错", "很好", "厉害"],
    }
    
    @classmethod
    def classify(cls, text: str) -> str:
        """
        分类意图
        """
        text = text.lower()
        
        for intent, keywords in cls.INTENTS.items():
            if any(kw in text for kw in keywords):
                return intent
        
        return "unknown"

# 演示意图识别
def demo_intent_classification():
    test_cases = [
        "你好，在吗？",
        "Python 是什么？",
        "请帮我查一下天气",
        "谢谢你的帮助！",
        "再见！",
        "今天天气不错啊",  # unknown
    ]
    
    print("=== 意图识别演示 ===\n")
    for text in test_cases:
        intent = IntentClassifier.classify(text)
        print(f"输入: {text}")
        print(f"意图: {intent}\n")

demo_intent_classification()

## 17.6 完整对话示例

In [ ]:
async def demo_conversation():
    """
    演示完整对话流程
    """
    print("=== 对话演示 ===\n")
    print("(模拟对话，不实际调用 LLM)\n")
    
    # 模拟对话
    conversations = [
        ("你好", "你好！我是你的 AI 助手。有什么我可以帮助你的吗？"),
        ("我叫小明", "你好小明！很高兴认识你。"),
        ("记住，我喜欢 Python", "好的小明，我记住了你喜欢 Python。"),
        ("我刚才说我叫什么名字？", "你告诉我你叫小明。"),
        ("我喜欢什么编程语言？", "你之前说喜欢 Python。"),
    ]
    
    # 简单记忆模拟
    memory = {}
    
    for user_msg, assistant_msg in conversations:
        print(f"用户: {user_msg}")
        
        # 简单信息提取
        if "我叫" in user_msg:
            memory["name"] = user_msg.replace("我叫", "").strip("。！？")
        elif "喜欢" in user_msg:
            memory["like"] = user_msg.split("喜欢")[1].strip("。！？")
        
        print(f"助手: {assistant_msg}")
        print(f"[记忆: {memory}]\n")

await demo_conversation()

## 17.7 常见面试问题

**Q: 对话 Agent 和 Chatbot 有什么区别？**
- A:
  | 特性 | 传统 Chatbot | 对话 Agent |
  |------|-------------|-----------|
  | 能力 | 规则/简单问答 | 复杂推理 |
  | 上下文 | 无/有限 | 完整记忆 |
  | 工具使用 | 无 | 可调用外部工具 |
  | 个性化 | 无 | 可记住用户偏好 |

**Q: 如何处理对话中的指代消解？**
- A:
  1. **明确化**：在 prompt 中要求 LLM 明确指代
  2. **上下文**：提供完整的对话历史
  3. ** Few-Shot**：提供指代消解的示例
  4. **后处理**：使用 NLP 工具检测和替换

**Q: 如何检测对话的结束？**
- A:
  1. **意图识别**：检测再见、结束等意图
  2. **话题检测**：用户没有新问题
  3. **满意度**：用户表达感谢/满意
  4. **超时**：一定时间无输入
  5. **主动确认**：询问「还有其他问题吗」

**Q: 如何处理多轮对话中的信息更新？**
- A:
  1. **时间戳**：记录信息的获取时间
  2. **版本控制**：保留最新版本
  3. **冲突检测**：检测矛盾信息
  4. **确认机制**：重要更新需要确认
  5. **置信度**：根据来源和次数评分

---

## 总结

本课程学习了对话 Agent 的核心知识：

| 知识点 | 说明 |
|--------|------|
| **核心要素** | 上下文、意图、实体、状态、回复、记忆 |
| **基础实现** | System Prompt + 消息历史 |
| **增强实现** | 短期记忆 + 长期记忆 |
| **状态管理** | 状态机追踪对话进程 |
| **意图识别** | 理解用户想要什么 |
| **指代消解** | 正确理解「它」「这个」等指代 |

**下一步**: L18 将学习多 Agent 协作！